<a href="https://colab.research.google.com/github/cristianmejia00/clustering/blob/main/Topic_Models_using_BERTopic_TOPIC_MODEL_20241101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic Modeling with BERTopic

🔴 copied from the [Kubota Colab](https://colab.research.google.com/drive/1YsDp5_qGXGJKsEXsS8DO8CA_lqZc6EpA).  

`Topic Models` are methods to automatically organize a corpus of text into topics.

Topic Model process:
1. Data preparation
2. Tranform text to numeric vectors
3. Multidimensionality reduction
4. Clustering
5. Topic analysis
6. Cluster assignation


This notebook uses the library `BERTopic` which is a one-stop solution for topic modeling including handy functions for plotting and analysis. However, BERTopic does not have a function to extract the X and Y coords from UMAP. If we need the coordinates then use the notebooks `Topic_Models_using_Transformers` instead. In any other situation, when a quick analysis is needed this notebook may be better.

This notebook is also the one needed for the heatmap codes included in this folder.

`BERTopic` is Python library that handles steps 2 to 6.
BERT topic models use the transformer architechture to generate the embeds (i.e. the vector or numeric representation of words) and are currently the state-of-the-art method for vectorization.

This notebook shows how to use it.

---
Reading:
[Topic Modeling with Deep Learning Using Python BERTopic](https://medium.com/grabngoinfo/topic-modeling-with-deep-learning-using-python-bertopic-cf91f5676504)
[Advanced Topic Modeling with BERTopic](https://www.pinecone.io/learn/bertopic/)


# Requirements

## Packages installation and initialization

In [ ]:
#!pip install bertopic[visualization]

In [37]:
import pandas as pd
import numpy as np
import time
import math
import uuid
import re
import os
import json
import pickle
from datetime import date
from itertools import compress
from bertopic import BERTopic
from umap import UMAP
#from gensim.parsing.preprocessing import remove_stopwords
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

## Connect your Google Drive

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [38]:
def find_e_keys(dictionary):
    # List comprehension to find keys starting with 'e'
    e_keys = [key for key in dictionary if str(key).lower().startswith('e')]
    return e_keys

# 🔴 Input files and options

Go to your Google Drive and create a folder in the root directory. We are going to save all related data in that directory.
Upload the dataset of news into the above folder.
- The dataset should be a `.csv` file.
- Every row in the dataset is a document
- It can any kind of columns. Some columns must contain the text we want to analyze. For example, a dataset of academic articles may contain a "Title" and/or "Abstract" column.

In [39]:
# The bibliometrics folder
# Colab
ROOT_FOLDER_PATH = "drive/MyDrive/Bibliometrics_Drive"

# Mac
ROOT_FOLDER_PATH = "/Users/cristian/Library/CloudStorage/GoogleDrive-cristianmejia00@gmail.com/My Drive/Bibliometrics_Drive"

# Change to the name of the folder where the dataset is uploaded inside the above folder
project_folder = 'Q339_igem'

analysis_id = 'a01_tm__f01_e01__hdbs'

# Filtered label
settings_directive = "settings_analysis_directive_2025-08-04-16-40.json"

In [40]:
# Read settings
with open(f'{ROOT_FOLDER_PATH}/{project_folder}/{analysis_id}/{settings_directive}', 'r') as file:
    settings = json.load(file)

In [41]:
# Input dataset
dataset_file_path = f"{ROOT_FOLDER_PATH}/{settings['metadata']['project_folder']}/{settings['metadata']['filtered_folder']}/dataset_raw_cleaned.csv"

In [42]:
# Function to save files
def save_as_csv(df, save_name_without_extension, with_index):
    "usage: `save_as_csv(dataframe, 'filename')`"
    df.to_csv(f"{ROOT_FOLDER_PATH}/{save_name_without_extension}.csv", index=with_index)
    print("===\nSaved: ", f"{ROOT_FOLDER_PATH}/{save_name_without_extension}.csv")

In [43]:
# prompt: a function to save object to a pickle file
def save_object_as_pickle(obj, filename):
  """
  Saves an object as a pickle file.

  Args:
      obj: The object to be saved.
      filename: The filename of the pickle file.
  """
  with open(filename, "wb") as f:
    pickle.dump(obj, f)


In [44]:
# prompt: a function to load pickle object given a path
def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


In [45]:
# Open the data file
df = pd.read_csv(f"{dataset_file_path}", encoding='latin-1')
print(df.shape)
df.head()

(4548, 14)


,X_N,uuid,UT,AU,Countries,PY,DI,ID,WC,Institutions,TI,AB,CR,C1
0,1,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,173,UNIPV-Pavia,ITA,2009,https://2009.igem.org/Team:UNIPV-Pavia,Food & Energy,NaN,UniversitÃ degli Studi di Pavia\r\r\nDipartim...,Ethanol? Whey not!,Cheese whey is classified as a special waste f...,NaN,NaN
1,2,f7173d50-7003-491e-a6ff-527fc1055e00,174,Newcastle,GBR,2009,https://2009.igem.org/Team:Newcastle,Environment,NaN,Newcastle University\r\r\nNewcastle upon Tyne\...,Bac-man: sequestering cadmium into Bacillus sp...,Cadmium contamination can be a serious problem...,NaN,NaN
2,3,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,175,TUDelft,NLD,2009,https://2009.igem.org/Team:TUDelft,Information Processing,NaN,"Delft University of Technology\r\r\nDelft, The...",Bacterial Relay Race,"In our project, we aim at creating a cell-to-c...",NaN,NaN
3,4,f0c10215-5236-419b-8112-635791483c0a,176,USTC,CHN,2009,https://2009.igem.org/Team:USTC,Foundational Advance,NaN,University of Science and Technology of China\...,E. coli Automatic Directed Evolution Machine: ...,Evolution is powerful enough to create everyth...,NaN,NaN
4,5,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,177,Warsaw,POL,2009,https://2009.igem.org/Team:Warsaw,Health & Medicine,NaN,University of Warsaw\r\r\nKrakowskie Przedmies...,BacInVader â a new system for cancer genetic...,The main aim of our project is to design a mod...,NaN,NaN




---



## PART 2: Topic Model

In [46]:
# bibliometrics_folder
# project_folder
# project_name_suffix
# ROOT_FOLDER_PATH = f"drive/MyDrive/{bibliometrics_folder}"

#############################################################
# Embeddings folder
embeddings_folder_name = settings['tmo']['embeds_folder']

# Which column has the year of the documents?
my_year = settings['tmo']['year_column']

# Number of topics. Select the number of topics to extract.
# Choose 0, for automatic detection.
n_topics = settings['tmo']['n_topics']

# Minimum number of documents per topic
min_topic_size = settings['tmo']['min_topic_size']

In [47]:
# Get the embeddings back.
embeddings = load_pickle(f"{ROOT_FOLDER_PATH}/{project_folder}/{settings['metadata']['filtered_folder']}/{embeddings_folder_name}/embeddings.pck")
corpus =     pd.read_csv(f"{ROOT_FOLDER_PATH}/{project_folder}/{settings['metadata']['filtered_folder']}/{embeddings_folder_name}/corpus.csv").reset_index(drop=True)

In [48]:
# Combine embeddings
documents = corpus.text.to_list()

In [49]:
# corpus['uuid'] = [uuid.uuid4() for _ in range(len(corpus.index))]
# corpus['X_N'] = [i for i in range(1, len(corpus.index)+1)]

In [50]:
len(documents)

4548

In [51]:
#len(embeddings) == len(documents)
len(embeddings['embeddings']) == len(documents)

True

In [52]:
from hdbscan.hdbscan_ import HDBSCAN
# Execute the topic model.
# I suggest changing the values marked with #<---
# The others are the default values and they'll work fine in most cases.
# This will take several minutes to finish.

# Initiate UMAP
umap_model = UMAP(n_neighbors=15,
                  n_components=5,
                  min_dist=0.0,
                  metric='cosine',
                  random_state=100)

if n_topics == 0:
  # Initiate topic model with HDBScan (Automatic topic selection)
  topic_model_params = HDBSCAN(min_cluster_size=min_topic_size,
                               metric='euclidean',
                               cluster_selection_method='eom',
                               prediction_data=True)
else:
  # Initiate topic model with K-means (Manual topic selection)
  topic_model_params = KMeans(n_clusters = n_topics)

# Initiate BERTopic
topic_model = BERTopic(umap_model = umap_model,
                       hdbscan_model = topic_model_params,
                       min_topic_size=min_topic_size,
                       #nr_topics=15,          #<--- Footnote 1
                       n_gram_range=(1,3),
                       language='english',
                       calculate_probabilities=True,
                       verbose=True)



# Footnote 1: This controls the number of topics we want AFTER clustering.
# Add a hashtag at the beggining to use the number of topics returned by the topic model.
# When using HDBScan nr_topics will be obtained after orphans removal, and there is no warranty that `nr_topics < HDBScan topics`.
# thus, with HDBScan `nr_topics` means N topics OR LESS.
# When using KMeans nr_topics can be used to further reduce the number of topics.
# We use the topics as returned by the topic model. So we do not need to activate it here.

In [53]:
# Compute topic model
#topics, probabilities = topic_model.fit_transform(documents, embeddings)
topics, probabilities = topic_model.fit_transform(documents, embeddings['embeddings'])

2026-03-10 14:15:37,548 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-10 14:15:51,165 - BERTopic - Dimensionality - Completed ✓
2026-03-10 14:15:51,167 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-10 14:15:53,295 - BERTopic - Cluster - Completed ✓
2026-03-10 14:15:53,299 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 14:15:55,971 - BERTopic - Representation - Completed ✓


In [54]:
topic_embeddings = topic_model.topic_embeddings_
print(len(topic_embeddings[0]))
print(len(topic_embeddings))

384
164


In [55]:
# Get the list of topics
# Topic = the topic number. From the largest topic.
#         "-1" is the generic topic. Genericr keywords are aggegrated here.
# Count = Documents assigned to this topic
# Name = Top 4 words of the topic based on probability
# Representation = The list of words representing this topic
# Representative_Docs = Documents assigned to this topic
tm_summary = topic_model.get_topic_info()
tm_summary

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1351,-1_the_of_and_to,"[the, of, and, to, in, we, for, is, our, by]",[Expanding the Synthetic Biology Toolkit for P...
1,0,185,0_pet_plastic_degradation_petase,"[pet, plastic, degradation, petase, plastics, ...",[Introducing PETase and MHETase into Alteromon...
2,1,83,1_light_control_of_in,"[light, control, of, in, the, cell, protein, t...",[Lights Camera Flip Engineering a LightActivat...
3,2,82,2_cancer_early_detection_mirna,"[cancer, early, detection, mirna, diagnosis, b...",[TFPI Methylation in Stool Samples for Early D...
4,3,78,3_biofilm_quorum_quorum sensing_sensing,"[biofilm, quorum, quorum sensing, sensing, bio...",[Quorum Sensing as a target mechanism for redu...
...,...,...,...,...,...
159,158,5,158_rdx_crvi_chromium_chlorophenol,"[rdx, crvi, chromium, chlorophenol, rdetox, bi...",[HexaTri Bioremediation of ChromiumVI Contamin...
160,159,5,159_mitochondrial_aerobic respiration_respirat...,"[mitochondrial, aerobic respiration, respirati...",[MitochONdriOFF Controlling Aerobic Respiratio...
161,160,5,160_nickel_nickel ions_ions_the nickel,"[nickel, nickel ions, ions, the nickel, rcn pr...",[Characterization of the rcn promoter for nick...
162,161,5,161_rice_of rice_maize_stress,"[rice, of rice, maize, stress, domain, starchb...",[RICAID China has a miserable memory of suffer...


In [56]:
# Save the topic model assets
tm_folder_path = f'{ROOT_FOLDER_PATH}/{project_folder}/{settings["metadata"]["analysis_id"]}'

if not os.path.exists(tm_folder_path):
  !mkdir $tm_folder_path

tm_summary.to_csv(f'{tm_folder_path}/topic_model_info.csv', index=False)

In [57]:
# Number of topics found
found_topics = max(tm_summary.Topic) + 1
found_topics

163

In [58]:
# Confirm all documents are assigned
sum(tm_summary.Count) == len(corpus)

True

In [59]:
# Get top 10 terms for a topic
topic_model.get_topic(0)

[('pet', np.float64(0.015003979186777084)),
 ('plastic', np.float64(0.014759533468372353)),
 ('degradation', np.float64(0.007060162971468204)),
 ('petase', np.float64(0.007040699653522315)),
 ('plastics', np.float64(0.00625625481975033)),
 ('microplastics', np.float64(0.005660255866915906)),
 ('waste', np.float64(0.005655355277763371)),
 ('the', np.float64(0.005330514449632307)),
 ('and', np.float64(0.005196802730255742)),
 ('pollution', np.float64(0.005128314432555195))]

In [60]:
# Get the top 10 documents for a topic
topic_model.get_representative_docs(0)

['Introducing PETase and MHETase into Alteromonas The abundance of polyethylene terephthalate PET plastics in the oceans cause issues for all types of life The sequential enzymes PETase and MHETase found in the soil bacterium Ideonella sakaiensis have shown degradation of PET plastics through metabolic activity However because I sakaiensis is found mainly in soil it is not an ideal host when attempting to degrade PET plastics in the ocean By introducing these genes into Alteromonas macleodii a marine bacterium there is the possibility of being able to degrade PET in saltwater conditions using its ability to form biofilm Using the Loop Assembly method plasmids were constructed containing the genes that encode for PETase and MHETase These plasmids were created in Escherichia coli and were subsequently conjugated into Alteromonas Ongoing experiments have shown the proper L inserts found in our Alteromonas cells and we predict these cells should be able to degrade PET plastics in marine co

In [61]:
# Others

# # Get the number of documents per topic (same as in the table above)
# topic_model.get_topic_freq(0)

# # Get the main keywords per topic
# topic_model.get_topics()

In [62]:
# Print the parameters used. (For reporting)
topic_model.get_params()

{'calculate_probabilities': True,
 'ctfidf_model': ClassTfidfTransformer(),
 'embedding_model': None,
 'hdbscan_model': HDBSCAN(prediction_data=True),
 'language': 'english',
 'low_memory': False,
 'min_topic_size': 5,
 'n_gram_range': (1, 3),
 'nr_topics': None,
 'representation_model': None,
 'seed_topic_list': None,
 'top_n_words': 10,
 'umap_model': UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=5, n_jobs=1, random_state=100, tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True}),
 'vectorizer_model': CountVectorizer(ngram_range=(1, 3)),
 'verbose': True,
 'zeroshot_min_similarity': 0.7,
 'zeroshot_topic_list': None}

In [63]:
tm_params = dict(topic_model.get_params())
for key, value in tm_params.items():
    tm_params[key]=  str(value)
with open(f'{tm_folder_path}/topic_model_params.json', 'w') as f:
    json.dump(tm_params, f, ensure_ascii=False, indent=4)
    print('Done')

Done


In [64]:
# Get the topic score for each paper and its assigned topic
topic_distr, _ = topic_model.approximate_distribution(documents, batch_size=1000)
distributions = [distr[topic] if topic != -1 else 0 for topic, distr in zip(topics, topic_distr)]

100%|██████████| 5/5 [00:04<00:00,  1.16it/s]


In [65]:
topic_distr

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4548, 163))

In [66]:
# Document information. Including the topic assignation
dataset_clustering_results = topic_model.get_document_info(documents, df = corpus, metadata={"Score": distributions})

# Check for orphans (Topic == -1), save and remove them
if -1 in dataset_clustering_results['Topic'].values:
    orphans = dataset_clustering_results[dataset_clustering_results['Topic'] == -1]
    orphans.to_csv(f'{tm_folder_path}/orphans.csv', index=False)
    dataset_clustering_results = dataset_clustering_results[dataset_clustering_results['Topic'] != -1]

In [67]:
# Standar format for report analysis
dataset_clustering_results = dataset_clustering_results.reset_index(drop=True)
dataset_clustering_results = dataset_clustering_results.drop(columns=['text'])
dataset_clustering_results['X_E'] = dataset_clustering_results['Score']
dataset_clustering_results['X_C'] = dataset_clustering_results['Topic'] + 1



# Assign 'level0' based on cluster coverage (90%)
cluster_counts = dataset_clustering_results['X_C'].value_counts().sort_values(ascending=False)
total_rows = len(dataset_clustering_results)
cumulative = cluster_counts.cumsum() / total_rows
main_clusters = cluster_counts.index[cumulative <= 0.9].tolist()
dataset_clustering_results['level0'] = dataset_clustering_results['X_C'].apply(lambda x: x if x in main_clusters else 99999)
dataset_clustering_results.head()


,UT,uuid,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Score,X_E,X_C,level0
0,173,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,Ethanol Whey not Cheese whey is classified as ...,52,52_milk_lactose_fl_whey,"[milk, lactose, fl, whey, lactoferrin, lactose...",[Probabyotics Infant Formula Probiotic HMOs Di...,milk - lactose - fl - whey - lactoferrin - lac...,0.082032,False,0.000000,0.000000,53,53
1,174,f7173d50-7003-491e-a6ff-527fc1055e00,Bacman sequestering cadmium into Bacillus spor...,48,48_cadmium_of cadmium_metal_heavy,"[cadmium, of cadmium, metal, heavy, heavy meta...",[ChocoCadmium Detecting Cadmium in Peruvian ca...,cadmium - of cadmium - metal - heavy - heavy m...,0.255795,False,1.000000,1.000000,49,49
2,177,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,BacInVader a new system for cancer genetic th...,5,5_cancer_tumor_cells_therapy,"[cancer, tumor, cells, therapy, microenvironme...",[Breast cancertargeting bacterial microrobots ...,cancer - tumor - cells - therapy - microenviro...,0.212192,False,0.000000,0.000000,6,6
3,179,3bd65477-8297-443f-b128-2859b0c57266,Prevention of Lifestyle Diseases Using Synthet...,16,16_cholesterol_obesity_cardiovascular_fatty,"[cholesterol, obesity, cardiovascular, fatty, ...",[Modifiyng microbes to degrade cholesterol to ...,cholesterol - obesity - cardiovascular - fatty...,0.056530,False,0.000000,0.000000,17,17
4,180,437ab4f7-b181-41e0-a5ff-597f8f10cd6f,EcolYMPIC GAMES The project exploits quorum se...,70,70_game_each_the game_dilemma,"[game, each, the game, dilemma, stories, priso...",[Bacterial Evolutionary Game Simulation BEGS f...,game - each - the game - dilemma - stories - p...,0.056329,False,0.786084,0.786084,71,71


In [68]:
# Standar format for report analysis
dataset_clustering_results['cl99'] = [True if x == 99 else False for x in dataset_clustering_results['level0']]
dataset_clustering_results['cl-99'] = [True if x == 99 else False for x in dataset_clustering_results['level0']]
dataset_clustering_results.head()

,UT,uuid,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Score,X_E,X_C,level0,cl99,cl-99
0,173,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,Ethanol Whey not Cheese whey is classified as ...,52,52_milk_lactose_fl_whey,"[milk, lactose, fl, whey, lactoferrin, lactose...",[Probabyotics Infant Formula Probiotic HMOs Di...,milk - lactose - fl - whey - lactoferrin - lac...,0.082032,False,0.000000,0.000000,53,53,False,False
1,174,f7173d50-7003-491e-a6ff-527fc1055e00,Bacman sequestering cadmium into Bacillus spor...,48,48_cadmium_of cadmium_metal_heavy,"[cadmium, of cadmium, metal, heavy, heavy meta...",[ChocoCadmium Detecting Cadmium in Peruvian ca...,cadmium - of cadmium - metal - heavy - heavy m...,0.255795,False,1.000000,1.000000,49,49,False,False
2,177,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,BacInVader a new system for cancer genetic th...,5,5_cancer_tumor_cells_therapy,"[cancer, tumor, cells, therapy, microenvironme...",[Breast cancertargeting bacterial microrobots ...,cancer - tumor - cells - therapy - microenviro...,0.212192,False,0.000000,0.000000,6,6,False,False
3,179,3bd65477-8297-443f-b128-2859b0c57266,Prevention of Lifestyle Diseases Using Synthet...,16,16_cholesterol_obesity_cardiovascular_fatty,"[cholesterol, obesity, cardiovascular, fatty, ...",[Modifiyng microbes to degrade cholesterol to ...,cholesterol - obesity - cardiovascular - fatty...,0.056530,False,0.000000,0.000000,17,17,False,False
4,180,437ab4f7-b181-41e0-a5ff-597f8f10cd6f,EcolYMPIC GAMES The project exploits quorum se...,70,70_game_each_the game_dilemma,"[game, each, the game, dilemma, stories, priso...",[Bacterial Evolutionary Game Simulation BEGS f...,game - each - the game - dilemma - stories - p...,0.056329,False,0.786084,0.786084,71,71,False,False


In [69]:
dataset_clustering_results.level0.value_counts()

level0
99999    328
1        185
2         83
3         82
4         78
        ... 
111       10
110       10
125        9
123        9
120        9
Name: count, Length: 118, dtype: int64

In [70]:
# Save the dataframe
dataset_clustering_results.to_csv(f'{tm_folder_path}/dataset_minimal.csv', index=False)

In [71]:
# Save the topic model
topic_model.save(f'{tm_folder_path}/topic_model_object.pck')

2026-03-10 14:16:02,106 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.




---

